## Mapping

In [ ]:
import pandas as pd
from src.config import *  
import re 

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
#Load metadata and raw data
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')
raw_data=pd.read_csv(DIR_CLEANED_DATA)

In [ ]:
# Read manual review by Mohamed
qa = pd.read_csv(DIR_META/'Benin QA Quantitative survey - Full Data with QA.csv')

In [ ]:
len(raw_data)

### Sanity Checks

In [ ]:
meta_columns=survey.name.tolist()
data_columns=raw_data.columns.tolist()

missing_in_meta=[col for col in data_columns if col not in meta_columns]
if missing_in_meta:
    print("Columns in raw data missing in metadata:", missing_in_meta)
missing_in_data=[col for col in meta_columns if col not in data_columns]
if missing_in_data:
    print("Columns in metadata missing in raw data:", missing_in_data)

### Prepare data for mapping
- convert columns to be mapped to string
- create mapping dictionary from choices dataframe


In [ ]:
def normalize_column(s: pd.Series) -> pd.Series:
    dtype = s.dtype

    if pd.api.types.is_float_dtype(dtype):
        return s.astype('Int64').astype('string')
    if pd.api.types.is_integer_dtype(dtype):
        return s.astype('Int64').astype('string')
    if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype):
        return s.astype('string').replace({'nan': pd.NA, '': pd.NA})
    
    return s


In [ ]:
mapping = {}
for list_name, group in choices.groupby('list_name'):
    mapping[list_name] = dict(zip(group['value'], group['label']))

In [ ]:
# Extract list names from 'type' column for select_one and select_multiple questions
selection_mask=survey['type'].str.startswith('select')
survey_select=survey.loc[selection_mask,["type","name","label", "hint"]].rename(columns={"name":"col_name"})
# Get the choices list names from type column
# (list_name can include digits, e.g. "likert_scale_10" -- don't drop them)
pattern = r'select_[a-z]*\s+([a-z0-9_]+)'
survey_select["list_name"]=survey_select['type'].apply(lambda x: re.search(pattern, str(x)).group(1))


In [ ]:
survey_select=survey_select[survey_select["col_name"].isin(raw_data.columns)]

In [ ]:
# Convert all mapping columns to string
cols_to_map=survey_select["col_name"].tolist()
for col in cols_to_map:
    raw_data[col]=normalize_column(raw_data[col])

### Mapping of the columns

In [ ]:
# Some columns contain lists of values separated by spaces; we need to split them before mapping
list_columns=survey_select[survey_select["type"].str.startswith('select_multiple')]["col_name"].tolist()
list_columns = [col for col in list_columns if col in survey_select["col_name"].tolist()]
single_value_columns = [col for col in cols_to_map if col not in list_columns]



In [ ]:
def map_select_multiple(column: pd.Series, mapping_dict: dict) -> pd.Series:
    def map_values(value: pd.Series) -> pd.Series:
        if pd.isna(value):
            return pd.NA
        else:
            values = str(value).split()
            mapped_values = [mapping_dict.get(v, v) for v in values]
            return '|'.join(mapped_values)
    
    return column.apply(map_values)

In [ ]:
# Apply mapping to all single value columns
mapped_data=raw_data.copy()
survey_select_single=survey_select[survey_select["col_name"].isin(single_value_columns)]
for col_name, list_name in zip(survey_select_single["col_name"],survey_select_single["list_name"]):
    mapped_data.loc[:,col_name]=mapped_data.loc[:,col_name].map(mapping[list_name])


In [ ]:
# Apply mapping to all list columns
survey_select_list=survey_select[survey_select["col_name"].isin(list_columns)]
for col_name, list_name in zip(survey_select_list["col_name"],survey_select_list["list_name"]):
    mapped_data.loc[:,col_name]=map_select_multiple(mapped_data.loc[:,col_name],mapping[list_name])     

###

In [ ]:
# Written-in stations (00_clean_data / src/stations.py) are single tokens with spaces as underscores,
# so map_select_multiple keeps them whole; turn them back into spaces now that mapping is done.
cols=["station_most_listened","station_listened_yesterday","station_past_7_days"]
mapped_data[cols] = mapped_data[cols].apply(lambda x: x.str.replace('_', ' '))

#### Correct department and location labels

In [ ]:
# keep raw version just in case
mapped_data['province_old'] = mapped_data['province']
mapped_data['location_old'] = mapped_data['location']

# create dictionary of keys and department names
dpt_dict = qa.set_index(['Submission Key'])['department name'].to_dict()

# repeat for location names
# create dictionary of keys and location names
loc_dict = qa.set_index(['Submission Key'])['location name'].to_dict()


# create functions
def map_province(x):
    try:
        dpt_dict[x]
    except KeyError:
        return None

def map_location(x):
    try:
        loc_dict[x]
    except KeyError:
        return None


# update province column with manual review
mapped_data['province'] = mapped_data['KEY'].map(dpt_dict).fillna(mapped_data['province_old'])

# update location column with manual review
mapped_data['location'] = mapped_data['KEY'].map(loc_dict).fillna(mapped_data['location_old'])


In [ ]:
# Houinta (geom_id d7f7f3c8a2cbcd3c5c71397ea658425b7e933ee6 in the sampling frame, tagged
# province=Oueme there) has a split QA tag: 6 of 24 respondents were individually QA-tagged
# Plateau, the other 18 Oueme. All 24 are GPS-confirmed the same physical settlement
# (within ~400m of each other, centroid 6.4551, 2.6234) -- one settlement can't legitimately
# sit in two departments, so this is a QA labeling inconsistency, not a real split cluster.
# User confirmed by lat/lon review (2026-08-19): treat this as one settlement and label it
# Oueme throughout, matching both the sampling frame and the QA majority.
n_before = (mapped_data.loc[mapped_data['location'] == 'Houinta', 'province'] == 'Plateau').sum()
mapped_data.loc[mapped_data['location'] == 'Houinta', 'province'] = 'Oueme'
print(f"Houinta: reset {n_before} respondent(s) from province=Plateau to province=Oueme "
      f"({(mapped_data['location'] == 'Houinta').sum()} total in this cluster)")


In [ ]:
mapped_data.loc[mapped_data.province_old != mapped_data.province].groupby(['province_old', 'province']).count()

In [ ]:
mapped_data.loc[mapped_data.location_old != mapped_data.location].groupby(['location_old', 'location']).count()

### Export mapped data

In [ ]:
mapped_data.to_csv(DIR_MAPPED_DATA, index=False)